# Day 1 · Module 1 — Disease Prediction (Clinical Risk Console)

**Objective:** *honestly evaluate* a clinical risk model — audit it for fairness and **defend a decision threshold**, not just report accuracy.

> **Instructor solution notebook** · kernel **AImed**. No gate here — if you get stuck, a CHECKPOINT cell shows the answer (a plot + caption).

### Pre-work recap
- AUROC (ranking) vs **AUPRC** (the one that matters under imbalance) vs calibration.
- *Arrive-with:* why can a 99%-accurate sepsis model be useless?
- Dataset: bundled **Diabetes** (442 pts; `sex` A/B is the protected attribute; `high_progression` label).

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import clinical_risk_console` and `from common import metrics`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## Guided hands-on

In [ ]:
# Guided hands-on — these helpers do the math; the EXERCISE is the DECISION cell below.
import clinical_risk_console as CC
from common import metrics as M
df, X, y, groups, feats = CC.load_data()    # df=table, X=features, y=0/1 label, groups=sex A/B
Xtr, Xte, ytr, yte, gtr, gte = CC.split(X, y, groups)   # stratified 70/30 split, fixed seed
model = CC.train_models(Xtr, ytr)['histgb']             # gradient-boosting risk model
proba = model.predict_proba(Xte)[:, 1]                  # P(progression) for each test patient
auprc = M.auprc(yte, proba)                             # ranking quality under imbalance
report = M.subgroup_report(yte, proba, gte)             # per-sex AUROC + calibration (ECE)
print('overall AUPRC =', round(auprc, 3))
for g, r in report.items():
    a = None if r['auroc'] is None else round(r['auroc'], 2)
    print('sex', g, '| n', r['n'], '| AUROC', a, '| ECE', round(r['ece'], 2))
# threshold table: how flag-rate / missed cases / false alarms move with the cutoff
for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    c = M.confusion_at(yte, proba, t)
    print('thr', t, '| flagged', round(float((proba >= t).mean()) * 100), '%',
          '| sens', round(c['sensitivity'], 2), '| FN', c['fn'], '| FP', c['fp'])

### Key terms — the metrics this module uses

All of these come from the four confusion counts (TP, FP, FN, TN) — the same ones behind the threshold tools below.

| Metric | Formula | The question it answers | Also called |
|---|---|---|---|
| **Recall** | TP / (TP + FN) | of patients who *truly* progress, how many did we catch? | sensitivity, TPR |
| **Precision** | TP / (TP + FP) | of the patients we *flagged*, how many truly progress? | PPV |
| **Specificity** | TN / (TN + FP) | of *truly healthy* patients, how many did we correctly clear? | TNR |

- **Precision vs recall is a trade-off.** Lower the threshold -> you flag more people -> recall goes UP (catch more real cases) but precision goes DOWN (more flags are false alarms). Raise the threshold -> the reverse.
- **AUROC** (threshold-free *ranking*): the chance a random truly-sick patient scores higher than a random healthy one. 0.5 = coin flip, 1.0 = perfect.
- **AUPRC** = area under the precision-recall curve = a threshold-free summary of the precision/recall trade-off. When positives are **rare**, AUPRC is the honest headline number — AUROC can look strong while precision is poor. (Our diabetes set is ~50/50, so here the two largely agree; the gap matters most under heavy imbalance.)
- **Calibration = are the probabilities honest?** Think of a **weather forecaster**: when the model says 0.7 it should be right about 70% of the time. **ECE (Expected Calibration Error)** puts a number on it — bin patients by predicted probability, in each bin compare the average prediction to the actual event rate, then average those gaps (weighted by bin size). **0 = perfectly calibrated; bigger = more dishonest probabilities.** A model can rank well (good AUROC) yet be badly calibrated (high ECE) — the two can come apart, so a fairness audit has to check **both** per subgroup. *Which* dimension the disparity actually lives in is what you'll uncover in the audit below.

In [ ]:
# --- Precision-recall curve (the paper's other axis) ---------------------------------
# recall = sensitivity (true cases caught); precision = of those flagged, the share truly sick.
# As the threshold sweeps high->low the model traces this curve; AUPRC is the area under it.
%matplotlib inline
from sklearn.metrics import precision_recall_curve
import numpy as np, matplotlib.pyplot as plt
prec, rec, thr = precision_recall_curve(yte, proba)
baseline = float(np.mean(yte))      # precision you'd get by flagging EVERYONE (no-skill line)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(rec, prec, color='C0', label=f'model (AUPRC = {auprc:.2f})')
ax.axhline(baseline, color='gray', ls='--', label=f'flag-everyone baseline = {baseline:.2f}')
ax.set(xlabel='recall = sensitivity (true cases caught)', ylabel='precision (flags that were right)',
       xlim=(0, 1), ylim=(0, 1.02), title='Precision-recall: the cost of catching more cases')
ax.legend(loc='lower left')
plt.tight_layout(); plt.show()
print('Read it: pushing toward high recall (right) costs precision (down) - more catches, more false alarms.')

In [ ]:
# --- Why imbalance makes accuracy (and the ROC) lie — the slide's thesis, on our data ----
# Our diabetes set is ~50/50, but real screening is IMBALANCED (few true positives). Resample
# the TEST set down to ~10% prevalence and watch each metric react.
import numpy as np
from common import metrics as M
rng = np.random.default_rng(0)
pos = np.where(yte == 1)[0]; neg = np.where(yte == 0)[0]
keep = rng.choice(pos, size=max(1, len(neg) // 9), replace=False)   # ~1 positive per 9 negatives
idx = np.concatenate([keep, neg])
y_imb, p_imb = yte[idx], proba[idx]
prev = float(y_imb.mean())
trivial_acc = 1.0 - prev          # accuracy of a model that ALWAYS predicts 'not progress'
print(f'balanced   : prevalence {yte.mean():.2f} | AUROC {M.auroc(yte, proba):.3f} | AUPRC {M.auprc(yte, proba):.3f}')
print(f'imbalanced : prevalence {prev:.2f} | AUROC {M.auroc(y_imb, p_imb):.3f} | AUPRC {M.auprc(y_imb, p_imb):.3f}')
print(f'--> trivial "always healthy" accuracy on the imbalanced set = {trivial_acc:.2f}  (high accuracy, ZERO skill)')
print('AUROC barely moves; AUPRC collapses toward the prevalence; accuracy becomes a liar.')

## Your turn — the clinic can't have everything

The model is going live in a clinic that can follow up **only ~5 of every 100 patients**. The clinical lead has one rule: **catch at least 90% of the patients who will truly progress.**

**First, predict — no code.** Can a *single* decision threshold satisfy both the clinical lead *and* the clinic's capacity? Jot down yes/no and one sentence why.

**Now find out.** Using the toolbox below, work out:
- the threshold that catches **≥90% of true cases** — and what **% of patients** it ends up flagging;
- how *low* you can push the **% flagged** toward the clinic's ~5% budget — can you even get there?

**Then decide.** If the two demands collide, you can ship only one. **Which constraint do you honor, which do you break, and how many real cases (FN) does your choice miss?** You'll defend it in the `DECISION` cell.

In [ ]:
# WORKED — the two demands collide.
import numpy as np
# (1) catch >=90% of true cases -> a LOW threshold that floods the clinic:
thr90, sens90, spec90 = M.threshold_at_sensitivity(yte, proba, 0.90)
print(f'catch >=90% of cases -> threshold {thr90:.2f}  (flags {(proba >= thr90).mean() * 100:.0f}% of patients)')
# (2) how low can % flagged go? sweep and find the floor:
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
flagged = np.array([(proba >= t).mean() * 100 for t in grid])
print(f'strictest cutoff {grid[-1]:.2f} still flags {flagged[-1]:.0f}%  ->  the ~5% budget is INFEASIBLE here')
print('Verdict: 90% sensitivity floods the clinic; 5% capacity is unreachable -> you cannot have both.')

## Stuck coding, can you fill out this table?

In [ ]:
# Fill the three values from `c`.
import numpy as np, pandas as pd
rows = []
for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
    c = M.confusion_at(yte, proba, t)          # everything at this one cutoff
    rows.append({
        'threshold':   t,
        'flagged_%':   round(float((proba >= t).mean()) * 100, 1),
        'sensitivity': round(c['sensitivity'], 2),
        'FN_missed':   c['fn'],
    })
threshold_table = pd.DataFrame(rows)
print(threshold_table.to_string(index=False))

In [ ]:
# To compare MANY cutoffs at once, make one row per threshold — the same threshold table
  # from the guided cell, now as a DataFrame you can sort, filter, or plot:
import pandas as pd
threshold_table = pd.DataFrame([
  {'threshold': t,
   'flagged_%': round(float((proba >= t).mean()) * 100, 1),
   'sensitivity': M.confusion_at(yte, proba, t)['sensitivity'],
   'FN_missed':  M.confusion_at(yte, proba, t)['fn'],
   'FP_alarm':   M.confusion_at(yte, proba, t)['fp']}
  for t in np.round(np.arange(0.05, 0.96, 0.05), 2)])
print(threshold_table.to_string(index=False))

### Don't feel like coding? Just pick a threshold and read its confusion matrix
Fill in `my_threshold` below (read a promising cutoff off the table above) and the cell prints the 2x2 confusion matrix at that threshold.

In [ ]:
from common import metrics as M
# (a) The clinical floor: the highest threshold that still catches >= 90% of true cases.
thr90, sens90, spec90 = M.threshold_at_sensitivity(yte, proba, 0.90)
print(f'threshold for >=90% sensitivity = {thr90:.2f}  (sens {sens90:.2f}, spec {spec90:.2f})')
# (b) The clinic wants <=5% flagged, but even the strictest cutoff over-flags. Ship the
#     nearest-to-budget cutoff and be honest that 5% is infeasible.
nearest = threshold_table.iloc[(threshold_table['flagged_%'] - 5).abs().idxmin()]
my_threshold = float(nearest['threshold'])
print(f"nearest-to-5% cutoff = {my_threshold:.2f}  (still flags {nearest['flagged_%']:.0f}%)")
c = M.confusion_at(yte, proba, my_threshold)
print()
print(f'confusion matrix @ threshold {my_threshold}')
print('              pred NEG   pred POS')
print(f'  actual NEG   {c["tn"]:7d}   {c["fp"]:7d}   (FP = false alarms)')
print(f'  actual POS   {c["fn"]:7d}   {c["tp"]:7d}   (FN = missed cases)')
print(f'  sensitivity {c["sensitivity"]:.2f} | specificity {c["specificity"]:.2f}')

In [ ]:
# The student notebook drives this with an interactive matplotlib threshold slider (needs the
# ipympl widget backend), which can't run headless — so here we read the same FN/FP tradeoff
# off a static sweep table:
for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    c = M.confusion_at(yte, proba, t)
    print('thr', t, '| flagged', round(float((proba >= t).mean()) * 100), '%',
          '| sens', round(c['sensitivity'], 2), '| spec', round(c['specificity'], 2))

### Your task — make and defend a clinical decision
The numbers above are the easy part. Using them, fill in the **`DECISION`** cell: which threshold would you ship for a screening program that can follow up only **~5% of patients**, and *why* (name the missed-case vs false-alarm tradeoff)? Which `sex` group does the model treat **unfairly**, and by what *mechanism* — is the gap in **ranking** (a per-group AUROC gap) or in **calibration** (a per-group ECE gap)? Compare the two `sex` rows from the guided cell and decide which dimension the disparity really lives in. There is no single right number — what matters is the reasoning, which you argue in the discussion.

### Feel free to create any visualizations or tables to help in the analysis.


In [ ]:
# A defensible answer (yours may differ — the REASONING is what matters):
worst = min(report, key=lambda g: (report[g]['auroc'] or 1.0))  # the group the model RANKS worst
DECISION = {
    'ship_threshold': my_threshold,
    'why': ('The ~5% budget is infeasible (even the strictest cutoff flags ~13%), so I ship the '
            'strictest defensible cutoff and flag the capacity shortfall to ops: we accept many '
            'missed cases (high FN) rather than pretend a 5% screen is reachable.'),
    'unfair_group': worst,
    'mechanism': ('A discrimination gap: the model RANKS one sex group worse (AUROC ~0.74 vs ~0.85) '
                  'while BOTH groups are similarly, imperfectly calibrated (ECE ~0.18) — the harm is '
                  'unequal ranking, not one group being uniquely miscalibrated.'),
}
print(DECISION)

### Checkpoint — stuck? Visualize the answer
Couldn't reach a confident decision? Run the next cell: it plots the **fairness audit** (AUROC vs ECE by `sex`, the ROC curves, and the calibration curves) plus the **screening tradeoff**, and the caption below tells you how to read your answer straight off the figure.

If you're still circling, walk these rungs:
1. At threshold 0.5, what % is flagged and what's the sensitivity?
2. Lower the threshold until sensitivity ≥ 0.90. What threshold is that — and what % is flagged now?
3. Your clinic can flag only ~5%. How close can any threshold even get?
4. Put rows 2 and 3 side by side. They don't match. Which do you ship, and why?

In [ ]:
# CHECKPOINT — stuck on the decision? This VISUALIZES the answer so you can read it off.
# (force static inline plots here — the slider cell above may have switched to the widget backend)
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from clinical_risk_console import run_report
from common import metrics as M
st = run_report(log=lambda *a: None)
proba = st['results']['histgb']['proba']; yte = st['yte']; gte = st['gte']
groups = np.unique(gte)
fig, ((axB, axR), (axC, axS)) = plt.subplots(2, 2, figsize=(11, 8))

# TOP-LEFT — verdict: ranking (AUROC) differs, calibration (ECE) does not
auroc = [M.auroc(yte[gte == g], proba[gte == g]) for g in groups]
ece = [M.expected_calibration_error(yte[gte == g], proba[gte == g]) for g in groups]
x = np.arange(2); w = 0.35
axB.bar(x - w / 2, auroc, w, label='AUROC (ranking)')
axB.bar(x + w / 2, ece, w, label='ECE (calibration)')
axB.set_xticks(x); axB.set_xticklabels([f'sex {g}' for g in groups])
axB.set_ylim(0, 1); axB.set_title('Ranking differs, calibration ~equal'); axB.legend()

# TOP-RIGHT — the ranking gap as a curve: the lower ROC = the worse-ranked group
axR.plot([0, 1], [0, 1], 'k--', lw=1)
for g, a in zip(groups, auroc):
    fpr, tpr, _ = roc_curve(yte[gte == g], proba[gte == g])
    axR.plot(fpr, tpr, label=f'sex {g} (AUROC={a:.2f})')
axR.set(xlabel='false-positive rate', ylabel='true-positive rate', title='ROC by subgroup')
axR.legend(loc='lower right')

# BOTTOM-LEFT — calibration is NOT the gap: both groups land ~on the diagonal together
axC.plot([0, 1], [0, 1], 'k--', lw=1, label='perfectly calibrated')
for g, e in zip(groups, ece):
    obs, pred = M.calibration_points(yte[gte == g], proba[gte == g], n_bins=5)
    axC.plot(pred, obs, marker='o', label=f'sex {g} (ECE={e:.2f})')
axC.set(xlabel='predicted probability', ylabel='observed frequency', title='Calibration: NOT the disparity')
axC.legend(loc='upper left')

# BOTTOM-RIGHT — screening tradeoff: the ~5% capacity line vs what the model can actually deliver
ts = np.linspace(0.05, 0.95, 25)
axS.plot(ts, [(proba >= t).mean() for t in ts], label='fraction flagged')
axS.plot(ts, [M.confusion_at(yte, proba, t)['sensitivity'] for t in ts], label='sensitivity')
axS.axhline(0.05, color='r', ls=':', label='5% follow-up capacity')
axS.set(xlabel='decision threshold', title='Screening tradeoff'); axS.legend()
plt.tight_layout(); plt.show()
print('Fairness: AUROC bars/ROC curves differ (ranking gap); ECE bars + calibration curves ~overlap')
print('          -> the disparity is in RANKING, not calibration.')
print('Screening: the blue curve never reaches the 5% line -> even the strictest cutoff over-flags;')
print('           5% capacity is infeasible for this model (floor ~13% flagged at thr 0.95).')

**How to read the checkpoint plot.**

*Top-left, top-right, bottom-left — is the model fair across `sex`?* A fair model must do two separate things equally well for each group: **rank** patients (catch the sick first → AUROC) and quote **honest probabilities** (→ ECE).

- **Top-left (bars).** The two **AUROC** bars are unequal (≈0.74 vs ≈0.85) — the model ranks one group's patients worse. The two **ECE** bars are about equal (≈0.18) — calibration is similar. So the disparity is in **ranking**: a **discrimination gap**, not a calibration gap.
- **Top-right (ROC).** The same fact as a curve — each group's ROC traces how well it separates sick from healthy; the **lower curve belongs to the worse-ranked group**. (We use ROC, not a precision-recall curve, because ROC doesn't move with a group's base rate — so a gap here is a *skill* gap, not a prevalence artifact.)
- **Bottom-left (calibration).** The honesty check (the weather-forecaster view): both `sex` curves land ~on the diagonal **together**, confirming calibration is **NOT** the disparity.

Your `unfair_group` is the one with the **lower AUROC bar / lower ROC curve**.

*Bottom-right — screening tradeoff (how to pick the threshold).* **Analogy — a clinic with only a few callback slots.** You can follow up just **5 out of every 100 patients** — the red capacity line.

- **Blue = the fraction of patients you would flag.** Raise the threshold and you flag fewer people — but notice the blue curve **never drops to the red 5% line**: even at the strictest cutoff (0.95) the model still flags ~13%. **So 5% capacity is infeasible for this model** — you either accept the strictest cutoff (~13% flagged, but low sensitivity), expand capacity, or get a better model.
- **Orange = sensitivity = the share of true cases you actually catch.** It collapses as the threshold rises: being pickier lets more real cases slip through.

So the tradeoff is brutal — chasing capacity forces a **high** threshold that misses most true cases, and you still can't reach 5%. Your `ship_threshold` is the most defensible point you can argue on this curve, not a number where blue meets red.

### Discussion (peer instruction)
- A model has 0.91 AUROC but predicts 0.6 for nearly everyone. Useful?
- **Different** AUROC across `sex` groups but *similar* calibration — is the model *fair*, and can one threshold serve both?
- Accuracy 97%, prevalence 3% — what is the trivial baseline?

### Stretch (optional)
Add SHAP attributions (`import shap`) and ask whether the top features are *causal* or merely correlated with care access.